In [1]:
# Common Libraries
import pandas as pd
import numpy as np
import datetime

import scipy.stats as stats

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

import warnings
from pathlib import Path

---
### Common Help Functions

In [2]:
# Helpers

# Convert time string to hours float
def time_to_hours(timestr: str) -> float | None:
    try:
        h, m, s = map(int, timestr.split(":"))
        return h + (m / 60) + (s / 3600)
    except Exception:
        return None

---
### Data Help Functions

In [3]:
# Data libraries
import requests
import json
from numpy.polynomial.polyutils import RankWarning

UTMB Tokens - Login for additional statistics

In [4]:
# Get UTMB Live Access Token [race results]
def get_utmb_live_access_token(utmb_username: str, utmb_password:str) -> str:

    login_data = {
        "grant_type": "password",
        "client_id": "utmb-world",
        "username": utmb_username,
        "password": utmb_password,
        "scope": "openid profile email"
    }

    res = requests.post("https://accounts.utmb.world/auth/realms/utmb-world/protocol/openid-connect/token", data=login_data)
    token_data = res.json()
    access_token = token_data["access_token"]

    return access_token

In [5]:
# Get UTMB API access token [runner results]

def get_utmb_access_token(utmb_username: str, utmb_password: str) -> str:

    url = "https://accounts.utmb.world/auth/realms/utmb-world/protocol/openid-connect/token"

    payload = {
        "grant_type": "password",
        "client_id": "utmb-world",
        "username": utmb_username,
        "password": utmb_password
    }

    res = requests.post(url, data=payload)
    data = res.json()
    access_token = data.get("access_token")
    
    return access_token

Get and Sort Data

In [6]:
# Get race results data

def get_race_results(race_id: str, course_id: str, race_year: int, access_token: str, printouts: bool = False) -> pd.DataFrame:
    print(f"Retrieving data for: {race_id} ~ {course_id} ~ {race_year}")
    if printouts:
        print("-----------------------------------------")


    # The UTMB API returns results page by page
    max_pages = 100 # Maximum number of pages to try
    results_per_page = 100 # How many runners the API returns per page

    # HTTP headers for UTMB API
    headers = {
        "Accept": "*/*", # Accepts any type of response
        "User-Agent": "Mozilla/5.0", # Pretend to be a normal browser
        "Origin": "https://live.utmb.world", # Tell the server request came from UTMB live site
        "Referer": "https://live.utmb.world/", # Tell server what page "linked" us here
        "X-Tenant": f"{race_id}_{race_year}", # UTMB API key
        "content-type": "application/json", # Expecting JSON data
        "Authorization": f"Bearer {access_token}", # Login to UTMB page
    }


    # Run through all pages
    race_results_list = []
    for i in range(max_pages + 1):
            
        # GET request to the UTMB API
        res = requests.get(
            f"https://utmblive-api.utmb.world/races/{course_id}/progressive",
            params={"type": "PROGRESSIVE_RANKING", "page": i, "limit": results_per_page},
            headers=headers,
        )

        # Fail early if API breaks
        res.raise_for_status()  

        data = res.json()
        runners = data.get("runners", [])

        # Stop when no runners left
        if not runners:
            if printouts:
                print(f"No more results at page {i} ~> {sum(len(x) for x in race_results_list)} results. Stopping.")
            break

        # flatten JSON to DataFrame
        df = pd.json_normalize(runners) 
        race_results_list.append(df)

        if printouts:
            print(f"Page {i} done ~> {sum(len(x) for x in race_results_list)} runners so far.")

    # Combine all pages into one DataFrame
    if not race_results_list:
        race_results = pd.DataFrame()

    else:
        warnings.filterwarnings(
            "ignore",
            message=".*DataFrame concatenation with empty or all-NA entries is deprecated.*",
            category=FutureWarning
            )
        
        race_results = (
            pd.concat(race_results_list, ignore_index=True)
            [[
                "raceId", "raceName", "raceCategory", "start",
                "info.fullname", "info.url", "info.index",
                "info.sex", "info.age", "info.countryCode", "info.category", "info.club",
                "ranking.scratch", "ranking.sex", "ranking.category",
                "raceTime", "diffToFirst", "status", "isFinisher"
            ]]
            .rename(columns={
                "raceId": "race_id", 
                "raceName": "race_name", 
                "raceCategory": "race_category",
                "start": "race_start_time",
                "info.fullname": "runner_name", 
                "info.url": "runner_url", 
                "info.index": "runner_pre_race_index",
                "info.sex": "runner_gender", 
                "info.age": "runner_age", 
                "info.countryCode": "runner_country_code", 
                "info.category": "runner_category", 
                "info.club": "runner_club",
                "ranking.scratch": "runner_rank", 
                "ranking.sex": "runner_rank_gender", 
                "ranking.category": "runner_rank_category",
                "raceTime": "runner_race_time", 
                "diffToFirst": "runner_diff_to_first_time", 
                "status": "runner_final_status", 
                "isFinisher": "runner_is_finisher"
            })
            .assign(
                age = lambda x: pd.to_numeric(x["runner_age"], errors="coerce"),
                runner_race_time_hours = lambda x: x["runner_race_time"].apply(time_to_hours),
                runner_diff_to_first_time_hours = lambda x: x["runner_diff_to_first_time"].apply(time_to_hours),
                race_start_time_hours = lambda x: x["race_start_time"].apply(time_to_hours),
                runner_final_status_map = lambda x: x["runner_final_status"].map({"f": "finisher", "a": "dnf", "hd": "broomed"})
            )

        )

    # Format data ...
    # race_results.loc[race_results["runner_is_finisher"] == False, "runner_race_time_hours"] = np.nan
    # race_results.loc[race_results["runner_is_finisher"] == False, "runner_race_time"] = np.nan
    
    race_results["runner_age"] = race_results["runner_age"].astype(float)

    return race_results

In [7]:
# Get runner results data

def get_runner_results(runner_id: str, access_token: str) -> pd.DataFrame:

    # Set up HTTP headers for UTMB API
    headers = {
        "Accept": "*/*", # Accepts any type of response
        "User-Agent": "Mozilla/5.0", # Pretend to be a normal browser
        "Origin": "https://utmb.world", # Tell the server request came from UTMB site
        "Referer": "https://utmb.world/", # Tell server what page "linked" us here
        "x-tenant-id": "worldseries", # UTMB API key
        "content-type": "application/json", # Expecting JSON data
        "Authorization": f"Bearer {access_token}" # Login to UTMB page
    }

                
    # GET request to the UTMB API
    res = requests.get(
        url=f"https://api.utmb.world/runners/{runner_id}/results",
        params={"page": 1, "limit": 100},
        headers=headers,
    )

    data = res.json()
    results = data.get("results", [])
    df = pd.DataFrame(results)

    runner_results_df = (
        df
        [[
            "dateIso", "race",  "uri", "piCategory", "utmbEventStatus", "country", "countryCode",
            "distance", "elevationGain", 
            "time", "isDnf", "rank", "rankGender", 
            "totalRanked", "totalRankedGender", "index",
        ]]
        .rename(columns = {
            "dateIso": "date",
            "race": "race_name",
            "uri": "race_uri",
            "utmbEventStatus": "utmb_event_status",
            "country": "country",
            "countryCode": "country_code",
            "distance": "distance",
            "elevationGain": "elevation_gain",
            "time": "race_time",
            "isDnf": "is_dnf",
            "rank": "rank",
            "rankGender": "rank_gender",
            "totalRanked": "total_finished",
            "totalRankedGender": "total_finished_gender",
            "index": "race_utmb_index",
        })
        .assign(
                distance = lambda x: x["distance"].apply(lambda x: pd.to_numeric(x, errors="coerce")),
                race_time_hours = lambda x: x["race_time"].apply(time_to_hours),
                date = lambda x: pd.to_datetime(x["date"]) 
        )
    )

    return runner_results_df

In [8]:
# Get runner additional pre-race statistics based on runner results data and race data

# Set of statistics
def get_runner_pre_race_statistics(runner_results_data: pd.DataFrame, race_date: pd.Timestamp) -> dict[str, float]:

    data = runner_results_data.copy()

    # New helper stats
    data["finished_rank_prop"] = (data["total_finished"] - (data["rank"]-1)) / data["total_finished"]
    data["finished_rank_prop_gender"] = (data["total_finished_gender"] - (data["rank_gender"]-1)) / data["total_finished_gender"]

    # Finished races data
    data_finished = data.loc[data["race_utmb_index"].notnull(), :].reset_index(drop=True)

    # Pre calculate heavier ones
    if not data_finished.empty:
        warnings.simplefilter('ignore', RankWarning)

        avg_finisher_rank_prop_gender_trend = np.polyfit(
            pd.to_datetime(data_finished["date"]).map(pd.Timestamp.toordinal), 
            data_finished["finished_rank_prop_gender"], 
            1)[0] * 365

        avg_finisher_rank_prop_trend = np.polyfit(
            pd.to_datetime(data_finished["date"]).map(pd.Timestamp.toordinal), 
            data_finished["finished_rank_prop"], 
            1)[0] * 365
        
        utmb_index_trend = np.polyfit(
            pd.to_datetime(data_finished["date"]).map(pd.Timestamp.toordinal), 
            data_finished["race_utmb_index"], 
            1)[0] * 365
        
    else:
        avg_finisher_rank_prop_gender_trend = 0
        avg_finisher_rank_prop_trend = 0
        utmb_index_trend = 0

    # Calculate stats
    stats = {
    "nr_races": data.shape[0],
    "days_from_last_race": (race_date.date() - data["date"].dt.date.max()).days if not data.empty else np.nan,
    "finish_prop": 1-data["is_dnf"].mean(),
    "avg_distance": data["distance"].mean(),
    "avg_elevation_gain": data["elevation_gain"].mean(),
    "avg_elevation_gain_distance_race": (data["elevation_gain"] / data["distance"]).mean(),

    "avg_race_index": data["race_utmb_index"].mean(),
    "sd_race_index": data["race_utmb_index"].std(),
    "utmb_index_trend": utmb_index_trend,

    "avg_finisher_rank_prop": data["finished_rank_prop"].mean(),
    "sd_finisher_rank_prop": data["finished_rank_prop"].std(),
    "avg_finisher_rank_prop_trend": avg_finisher_rank_prop_trend,

    "avg_finisher_rank_prop_gender": data["finished_rank_prop_gender"].mean(),
    "sd_finisher_rank_prop_gender": data["finished_rank_prop_gender"].std(),
    "avg_finisher_rank_prop_gender_trend": avg_finisher_rank_prop_gender_trend,

    "avg_pace": ((data["race_time_hours"] * 60) / data["distance"]).mean(),
    "avg_elevation_gain_adjusted_pace": ((data["race_time_hours"] * 60) / (data["distance"] + (data["elevation_gain"] / 100))).mean(),
    "avg_vertical_speed": (data["elevation_gain"] / data["race_time_hours"]).mean(),

    }

    return stats


# Data based sets of statisticss
def add_runner_additional_statistics(single_race_runner_data: pd.Series, utmb_access_token: str) -> pd.DataFrame:

    # Runner ID & Race Date
    single_race_runner_id = single_race_runner_data["runner_url"].split("runner/")[1]
    single_race_date = pd.to_datetime(single_race_runner_data["race_start_time"])

    runner_full_results = get_runner_results(
        runner_id = single_race_runner_id, 
        access_token = utmb_access_token
        )

    # Filter Masks & Helper functions
    date_masks = {
        "alltime": runner_full_results["date"].dt.date < single_race_date.date(),
        "last2years": runner_full_results["date"].dt.date >= (single_race_date.date() - datetime.timedelta(days=2*365))
    }

    distance_categories_filters = {
        "alldist": lambda df: df, 
        "100m": lambda df: df[df["piCategory"] == "100m"],
        "100k": lambda df: df[df["piCategory"] == "100k"],
        "sub100k": lambda df: df[~df["piCategory"].isin(["100m", "100k"])],
    }

    # Get runner additional statistics
    runner_additional_statistics = {}
    for datetime_prefix, date_mask in date_masks.items():
        for distance_prefix, distance_filter in distance_categories_filters.items():

            single_combination_runner_stats = get_runner_pre_race_statistics(
                runner_results_data =  distance_filter(runner_full_results[date_mask]), 
                race_date = single_race_date
                )
            
            runner_additional_statistics.update({
                datetime_prefix + "_" + distance_prefix + "_" + key: value 
                for key, value in single_combination_runner_stats.items()
                })
            
    runner_race_id = single_race_runner_data[["runner_url", "race_id", "race_name", "race_start_time"]].to_dict()
    runner_additional_statistics = runner_race_id | runner_additional_statistics

    return pd.Series(runner_additional_statistics)

---
### Get Data

In [9]:
# Config
utmb_username = "pecek.urh@gmail.com"
utmb_password = "PajkoBatujko**1989"
utmb_results_years = [2022, 2023, 2024, 2025] 

data_path = Path("data")
utmb_race_results_path = data_path / "utmb_race_results" 
utmb_runner_statistics_path = data_path / "utmb_runner_statistics"

Get Data

In [10]:
# Get and Save UTMB races data (If needed)

utmb_live_access_token = get_utmb_live_access_token(utmb_username, utmb_password)
for year in utmb_results_years:

    file_path = utmb_race_results_path / f"utmb_{year}_results.csv"
    if file_path.exists():
        print(f"File {file_path} already exists. Skipping download.")
        continue

    single_year_utmb_results = get_race_results(
        race_id = "utmb", 
        race_year = year, 
        course_id = "utmb", 
        access_token = utmb_live_access_token, 
        printouts = False
        )
    
    single_year_utmb_results.to_csv(f"data/utmb_race_results/utmb_{year}_results.csv", index=False)

File data\utmb_race_results\utmb_2022_results.csv already exists. Skipping download.
File data\utmb_race_results\utmb_2023_results.csv already exists. Skipping download.
File data\utmb_race_results\utmb_2024_results.csv already exists. Skipping download.
File data\utmb_race_results\utmb_2025_results.csv already exists. Skipping download.


In [11]:
# Import UTMB data and store it into a dictionary

race_results_data_dict = {
    year: pd.read_csv(utmb_race_results_path / f"utmb_{year}_results.csv")
    for year in utmb_results_years
}


In [12]:
# Get and Save Runner pre-race UTMB statistics data (If needed)

utmb_access_token = get_utmb_access_token(utmb_username, utmb_password)
for year in utmb_results_years:

    file_path = utmb_runner_statistics_path / f"utmb_{year}_runner_statistics.csv"
    if file_path.exists():
        print(f"File {file_path} already exists. Skipping download.")
        continue

    # Get runners additional statistics - runner by runner
    single_year_runner_race_utmb_additional_statistics = []
    for idx, row in single_year_utmb_results.iterrows():
        if idx % 50 == 0:
            print(f"processed row {idx} of {single_year_utmb_results.shape[0]} [year = {year}]...")
        try:
            row_stats = add_runner_additional_statistics(row, utmb_access_token)
            single_year_runner_race_utmb_additional_statistics.append(row_stats)
        except: 
            print(f"Failed for runner {row['runner_name']} ... [year {year}]")

    # Save single year data
    pd.DataFrame(single_year_runner_race_utmb_additional_statistics).to_csv(file_path, index=False)

File data\utmb_runner_statistics\utmb_2022_runner_statistics.csv already exists. Skipping download.
File data\utmb_runner_statistics\utmb_2023_runner_statistics.csv already exists. Skipping download.
File data\utmb_runner_statistics\utmb_2024_runner_statistics.csv already exists. Skipping download.
File data\utmb_runner_statistics\utmb_2025_runner_statistics.csv already exists. Skipping download.


In [13]:
# Import Runner UTMB pre-race Statistics data and store it into a dictionary

runner_statistics_data_dict = {
    year: pd.read_csv(utmb_runner_statistics_path / f"utmb_{year}_runner_statistics.csv")
    for year in utmb_results_years
}


Combine data

In [14]:
# Combiner Race Results and Runner Statistics data into a single dictionary (year by year)
race_results_runner_statistics_data_dict = {
    year: pd.merge(
        left = race_results_data_dict[year], 
        right = runner_statistics_data_dict[year], 
        on =["race_id", "race_name", "race_start_time", "runner_url"], 
        how="left"
    )
    for year in race_results_data_dict.keys()
}

In [15]:
# Combine all years into a single DataFrame

race_results_runner_statistics_data = pd.concat(
    [
        df.assign(race_year=year)[["race_year"] + [col for col in df.columns]] 
        for year, df in race_results_runner_statistics_data_dict.items()],
    ignore_index=True
)